# 02 - Unsupervised Regimes

Goal: group similar market days without using trading labels.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import silhouette_score

from nvda_rl.features import CLUSTER_FEATURE_COLUMNS, describe_regimes, fit_unsupervised_train_test

sns.set_theme(style="whitegrid", context="notebook")
data = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "nvda_features.csv", parse_dates=["date"])

## How k-means works

K-means tries to place days into groups where days inside one group look similar.

It starts with cluster centers, assigns each day to the closest center, then moves centers and repeats.

We scale features first because RSI, returns, and volume z-score use different units.

In [ ]:
split_date = "2021-01-01"
train_enriched, test_enriched, regime_model, pca_model = fit_unsupervised_train_test(
    data,
    split_date=split_date,
    n_clusters=3,
)
enriched = pd.concat([train_enriched, test_enriched], ignore_index=True).sort_values("date").reset_index(drop=True)
scaled_features = regime_model.named_steps["scaler"].transform(train_enriched[CLUSTER_FEATURE_COLUMNS])
score = silhouette_score(scaled_features, train_enriched["regime"])
print(f"Silhouette score: {score:.3f}")

regime_summary = describe_regimes(enriched)
regime_summary

The models are fitted only on the training period before 2021. Test rows are transformed with the saved scaler, K-means centers, and PCA components.

Regimes are interpreted with next-day return, not the same return used to create features. This avoids mechanically labeling clusters by today's return.

In [ ]:
regime_counts = enriched["regime"].value_counts().sort_index()
regime_counts.plot(kind="bar", figsize=(7, 4), color="tab:blue")
plt.title("Number of days in each regime")
plt.xlabel("Regime")
plt.ylabel("Days")
plt.tight_layout()
plt.show()

Regime 0 has the most days, so most trading days are normal positive days. Regime 1 is less frequent but has much stronger average return.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(enriched["date"], enriched["adj_close"], color="black", linewidth=1)
axes[0].scatter(enriched["date"], enriched["adj_close"], c=enriched["regime"], cmap="Set2", s=10, alpha=0.8)
axes[0].set_title("NVDA price colored by regime")
axes[0].set_ylabel("Adjusted close")

axes[1].scatter(enriched["date"], enriched["return"], c=enriched["regime"], cmap="Set2", s=10, alpha=0.7)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title("Daily returns colored by regime")
axes[1].set_ylabel("Daily return")
plt.tight_layout()
plt.show()

The stress regime appears around large price disruptions and drawdowns. This is useful because the RL agent can treat those days differently.

## How PCA works

PCA rotates the feature space into new axes that keep as much variation as possible.

We use two PCA columns as compact market-state signals for RL.

In [ ]:
explained = pca_model.named_steps["pca"].explained_variance_ratio_
print(f"PC1 explained variance: {explained[0]:.2%}")
print(f"PC2 explained variance: {explained[1]:.2%}")

plt.figure(figsize=(8, 6))
sns.scatterplot(data=enriched, x="pca_1", y="pca_2", hue="regime", palette="Set2", s=25, alpha=0.75)
plt.title("PCA market-state map")
plt.tight_layout()
plt.show()

The PCA chart shows that regimes occupy different areas of the feature map, so the clusters are not just random labels.

In [ ]:
transition = pd.crosstab(
    enriched["regime"].shift(1).rename("previous"),
    enriched["regime"].rename("current"),
    normalize="index",
)
transition

The transition table shows whether regimes persist. Persistence matters because a regime that lasts more than one day is easier to trade.

In [ ]:
output_path = PROJECT_ROOT / "data" / "processed" / "nvda_regimes.csv"
enriched.to_csv(output_path, index=False)
enriched[["date", "return", "volatility_20d", "rsi_14", "regime", "pca_1", "pca_2"]].head()

Output:

`nvda_regimes.csv` is the bridge from unsupervised learning to the RL notebooks. The file keeps the chronological train/test boundary honest: unsupervised models learn from train and only transform test.